# QueryFirst fiew visits

- author : Sylvie Dagoret-Campagne
- creation date : 2026-03-28
- affiliation : IN2P3/CNRS
- bps file : `00_run_bps_pipeline.sh`

```bash
bps submit bps_generic_main.yaml \
    -b /repo/main \
    -i LSSTCam/defaults \
    -o u/dagoret/test_runbps_2026_test_lc_fromalertsFink_v1 \
    -p ${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml#stage1-single-visit,stage2-recalibrate \
    -d "instrument='LSSTCam' AND skymap='lsst_cells_v2' AND visit IN (2026021600106, 2026021800057, 2026021800059, 2026021800061) AND detector IN (50, 53, 57, 60)"
```

https://tigress-web.princeton.edu/~lkelvin/pipelines/current/drp_pipe/LSSTCam/DRP/

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401
    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
from lsst.daf.butler import Butler
from lsst.geom import SpherePoint, degrees

import lsst.afw.display as afwDisplay
from firefly_client import FireflyClient
import firefly_client.plot as ffplt

In [ ]:
FLAG_DUMP_COLLECTIONS = True
FLAG_DEBUG_VISITS = True

In [ ]:
# ── Butler repository and DRP collection ──────────────────────────────────────
# Available DRP collections (uncomment the desired one):
#   'LSSTCam/runs/DRP/v30_0_1/DM-54061'
#   'LSSTCam/runs/DRP/v30_0_4/DM-54249'   <- currently selected
#   'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881'
REPO = 'main'
LSST_COLLECTIONS = ['LSSTCam/defaults']
MY_COLLECTIONS = ["u/dagoret/2026_test_lc_fromalertsFink"]
COLLECTIONS  =  LSST_COLLECTIONS + MY_COLLECTIONS 
skymapName = "lsst_cells_v2"
instrument = "LSSTCam"

# ── Date range for exposure queries ───────────────────────────────────────────
date_start = 20250401      # YYYYMMDD — earliest day_obs to include
date_selection = 20250401  # YYYYMMDD — reference date for the analysis

# ── Build WHERE clauses for Butler registry queries ───────────────────────────
where_clause = "instrument = '" + f"{instrument}" + "'"
where_clause_date = where_clause + f" and day_obs >= {date_start}"

# ── Instantiate the Butler and its registry ───────────────────────────────────
butler = Butler(REPO, collections=COLLECTIONS)
registry = butler.registry
print("Butler OK")

#### Dump collections

In [ ]:
if FLAG_DUMP_COLLECTIONS:
    # Lister toutes les collections existantes dans ce repo
    all_collections = butler.registry.queryCollections()
    #print("Toutes les collections dans le repo:")
    #for c in all_collections:
    #    print(c)

    # Filtrer celles qui contiennent "dagoret"
    dagoret_collections = [c for c in all_collections if "dagoret" in c]
    print("\nCollections dagoret trouvées:")
    for c in dagoret_collections:
        print(c)

    # Exemple d'utilisation sûre
    COLLECTIONS = ["LSSTCam/defaults"] + dagoret_collections

    skymapName = "lsst_cells_v2"
    skymap = butler.get("skyMap", skymap=skymapName, collections=COLLECTIONS)
    print("SkyMap récupéré !")

#### Load skymap

In [ ]:
# Retrieve the sky tessellation (skyMap) from the butler.
# The 'lsst_cells_v2' skymap divides the full sky into tracts and patches.
# Error handling is included because the skymap may not be present in all collections.
try:
    skymap = butler.get("skyMap", skymap=skymapName, collections=COLLECTIONS)
except Exception as inst:
    print(type(inst))   # exception type
    print(inst.args)    # arguments stored in .args
    print(inst)         # full message via __str__

In [ ]:
FLAG_DUMP_DATASETS = True
if FLAG_DUMP_DATASETS:
    #Il parcourt tous les dataset types et garde uniquement ceux qui existent réellement dans 
    # tes collections.
    for datasetType in registry.queryDatasetTypes():
        if registry.queryDatasets(datasetType, collections=COLLECTIONS).any(
            execute=False, exact=False
        ):
            # Skip pipeline bookkeeping products
            if (
                ("_config"         not in datasetType.name)
                and ("_log"        not in datasetType.name)
                and ("_metadata"   not in datasetType.name)
                and ("_resource_usage" not in datasetType.name)
                and ("Plot"        not in datasetType.name)
                and ("Metric"      not in datasetType.name)
                and ("metric"      not in datasetType.name)
            ):
                print(datasetType)


### Search image
- Graph : https://tigress-web.princeton.edu/~lkelvin/pipelines/current/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage1-single-visit.pdf

```
postISRCCD                                dims=['band', 'instrument', 'day_obs', 'detector', 'group', 'physical_filter', 'exposure']  present=True
post_isr_image                            dims=['band', 'instrument', 'day_obs', 'detector', 'group', 'physical_filter', 'exposure']  present=False
calexp                                    dims=['band', 'instrument', 'day_obs', 'detector', 'physical_filter', 'visit']  present=True
calexpBackground                          dims=['band', 'instrument', 'day_obs', 'detector', 'physical_filter', 'visit']  present=True
preliminary_visit_image                   dims=['band', 'instrument', 'day_obs', 'detector', 'physical_filter', 'visit']  present=False
preliminary_visit_image_background        dims=['band', 'instrument', 'day_obs', 'detector', 'physical_filter', 'visit']  present=False
```

In [ ]:
DATASET_IMG_POSTISR1  = "postISRCCD"
DATASET_IMG_POSTISR2  = "post_isr_image"  #
DATASET_IMG_CALEXP    = "calexp"
DATASET_IMG_CALEXPBG  = "calexpBackground"
DATASET_IMG_PRELIMVIS = "preliminary_visit_image"  
DATASET_IMG_PRELIMBG  = "preliminary_visit_image_background"  


# Print actual dimensions for confirmation
# Ce code sert à identifier les dataset types réellement peuplés dans tes collections Butler, sans lancer de requêtes coûteuses.
for dt_name in [DATASET_IMG_POSTISR1,DATASET_IMG_POSTISR2,
                DATASET_IMG_CALEXP,DATASET_IMG_CALEXPBG, 
                DATASET_IMG_PRELIMVIS, DATASET_IMG_PRELIMBG]:
    try:
        dt = registry.getDatasetType(dt_name)
        in_col = registry.queryDatasets(dt_name, collections=COLLECTIONS).any(execute=False, exact=False)
        print(f"{dt_name:40s}  dims={list(dt.dimensions.names)}  present={in_col}")
    except Exception as exc:
        print(f"{dt_name:40s}  ERROR: {exc}")


### Search for a visit table

In [ ]:
visitTable_name0 = "preliminary_visit_table"
visitTable_pattern1 = "*isitTable*"
visitTable_pattern2 = "*isitTable"
visitTable_name = "preVisitTable"

In [ ]:
if FLAG_DEBUG_VISITS:
    dataset_types = registry.queryDatasetTypes(visitTable_name0)
    for dt in dataset_types:
        print(dt.name)


In [ ]:
if FLAG_DEBUG_VISITS:
    dataset_types = list(butler.registry.queryDatasetTypes(visitTable_pattern1))

    # Pour un affichage lisible (nom du type et storage class)
    for dt in sorted(dataset_types, key=lambda x: x.name):
        print(f"{dt.name:20} | {dt.storageClass.name}")


In [ ]:
if FLAG_DEBUG_VISITS:
    dataset_types = list(butler.registry.queryDatasetTypes(visitTable_pattern2))
    for dt in dataset_types:
        print(f"{dt.name} :::", dt)
        print(f"    required dimensions: {dt.dimensions.required}")
        print()


### Search for sources/objects

```
src                                       dims=['band', 'instrument', 'day_obs', 'detector', 'physical_filter', 'visit']  present=True
object                                    dims=['skymap', 'tract']  present=False
source                                    dims=['band', 'instrument', 'day_obs', 'detector', 'physical_filter', 'visit']  present=False
object_forced_source                      dims=['skymap', 'tract', 'patch']  present=False
```

In [ ]:
# Hard-coded after discovery: confirmed present in DM-54249
DATASET_TYPE_PRESRC = "src"  # 
DATASET_TYPE_OBJ = "object"  # dims: skymap, tract
DATASET_TYPE_SRC = "source"  # dims: skymap, tract
DATASET_TYPE_FORCED = "object_forced_source"  # dims: skymap, tract, patch


# Print actual dimensions for confirmation
for dt_name in [ DATASET_TYPE_PRESRC, DATASET_TYPE_OBJ, DATASET_TYPE_SRC, DATASET_TYPE_FORCED]:
    try:
        dt = registry.getDatasetType(dt_name)
        in_col = registry.queryDatasets(dt_name, collections=COLLECTIONS).any(execute=False, exact=False)
        print(f"{dt_name:40s}  dims={list(dt.dimensions.names)}  present={in_col}")
    except Exception as exc:
        print(f"{dt_name:40s}  ERROR: {exc}")


In [ ]:
assert False

In [ ]:
afwDisplay.setDefaultBackend("firefly")

In [ ]:
idviz=0
for numvis, id_visit in enumerate(list_of_visits):
    dataId = {
        "instrument": "LATISS",  # instrument name
        "detector": 0,           # detector number (usually 0 for LATISS)
        "exposure": id_visit,  # your visit/exposure ID
        }

    try:
        idviz +=1 
        post_isr_image = butler.get("postISRCCD", dataId)
        meta = post_isr_image.getMetadata()
        #md = meta.toDict()
        #print(md)
        
        afw_display = afwDisplay.Display(frame=idviz)
        title = f"{numvis}) :: {id_visit}"
        

        afw_display.setImageColormap("gray")
        afw_display.scale("asinh", "zscale")
        afw_display.image(post_isr_image,title=title)
        

    except Exception as inst:
        print(type(inst))    # the exception type
        print(inst.args)     # arguments stored in .args
        print(inst)          # __str__ allows 
    
    

In [ ]:
#afw_display.clearViewer()